# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mansi-cs/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/mansi-cs/flyrank-ml-internship.git
%cd flyrank-ml-internship
!ls
!ls data/raw

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 147, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 147 (delta 56), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (147/147), 1.85 MiB | 9.17 MiB/s, done.
Resolving deltas: 100% (56/56), done.
/content/flyrank-ml-internship
AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission
content_refresh_anonymized.csv


In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
print(df.columns.tolist())

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

*A page should be refreshed when it is old,
still receives meaningful search visibility, and has weak CTR relative to its opportunity.*

In [3]:
# Signal Check 1: Staleness

stale_table = (
    df.groupby("freshness_tier")
      .agg(
          n=("content_id","count"),
          avg_impressions=("impressions_90d","mean"),
          avg_clicks=("clicks_90d","mean")
      )
      .round(2)
)

print(stale_table)

                    n  avg_impressions  avg_clicks
freshness_tier                                    
0-30            20480          4199.61       13.73
181+              174          1172.45        2.67
31-90             175          6506.75        9.69
91-180           9171          7486.67       21.77


### verdict: CONFIRMED
Content older than 180 days shows substantially lower impressions and clicks than fresher content.

In [4]:
#Signal Check 2: CTR vs Position
temp = df[df["avg_position"] > 0].copy()

ctr_position = (
    temp.groupby("position_tier")
        .agg(
            n=("content_id","count"),
            avg_ctr=("ctr","mean")
        )
        .round(3)
)

print(ctr_position)

                   n  avg_ctr
position_tier                
deep            1319    0.150
page_1         11814    0.652
page_3_5        7242    0.222
striking        7304    0.323
top_3           1116    2.764


### Verdict : confirmed
Higher-ranking pages receive significantly higher CTR, confirming that CTR and position are strongly related.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [20]:
import os

print(os.path.exists("work"))
print(os.path.exists("work/outputs"))
import os

os.makedirs("work/outputs", exist_ok=True)

print("Created:", os.path.exists("work/outputs"))

True
True
Created: True


In [21]:
import numpy as np

# Rule signals
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
low_ctr = (df["ctr"] < df["ctr"].median()).astype(int)

# Score
df["baseline_action_score"] = (
    stale * 2
    + visible * 2
    + low_ctr
)

# Reason codes
df["reason_code"] = np.select(
    [
        (stale == 1) & (visible == 1) & (low_ctr == 1),
        (stale == 1) & (visible == 1),
        (visible == 1) & (low_ctr == 1)
    ],
    [
        "stale_visible_lowctr",
        "stale_visible",
        "visible_lowctr"
    ],
    default="low_priority"
)

# Action labels
df["action_label"] = np.where(
    df["baseline_action_score"] >= 4,
    "refresh_content",
    "monitor"
)

# Rank queue
queue = df.sort_values(
    "baseline_action_score",
    ascending=False
)

# Export CSV
queue.to_csv(
    'work/outputs/baseline_action_score.csv',
    index=False
)

print("baseline_action_score.csv saved")

baseline_action_score.csv saved


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
queue.head(10)[[
    "content_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "impressions_90d",
    "ctr",
    "days_since_last_update"
]]

,content_id,baseline_action_score,reason_code,action_label,impressions_90d,ctr,days_since_last_update
698,content_b16bd7307b39,5,stale_visible_lowctr,refresh_content,4590,0.00,194
3507,content_074ba6ead17b,5,stale_visible_lowctr,refresh_content,533,0.00,183
11489,content_5feee3994adb,5,stale_visible_lowctr,refresh_content,7812,0.01,194
21268,content_0a91db491d14,4,stale_visible,refresh_content,13299,0.49,193
23215,content_bdbec75c1148,4,stale_visible,refresh_content,1316,0.15,194
5327,content_fe16a55cd13d,4,stale_visible,refresh_content,4556,0.33,194
26799,content_77d4d5930e5e,4,stale_visible,refresh_content,828,0.24,194
7021,content_1bfaa38ff26c,4,stale_visible,refresh_content,25715,0.23,194
12045,content_c2d929d83eaa,4,stale_visible,refresh_content,7558,0.20,193
22872,content_e3ff1b093148,4,stale_visible,refresh_content,1408,0.28,183


| Rank | Action          | Why it's there                                           | What would make it wrong                              |
| ---- | --------------- | -------------------------------------------------------- | ----------------------------------------------------- |
| 1    | refresh_content | Very old content (194 days), visible page, CTR 0.00%     | Content was recently updated but metadata is stale    |
| 2    | refresh_content | Old page with impressions but no clicks                  | Search intent may no longer be relevant               |
| 3    | refresh_content | High visibility (7812 impressions) and extremely low CTR | SERP features may be absorbing clicks                 |
| 4    | refresh_content | Large impression volume (13299) and stale content        | Seasonal traffic fluctuations                         |
| 5    | refresh_content | Old page and weak CTR                                    | Topic may have naturally declining interest           |
| 6    | refresh_content | High impressions and outdated content                    | Recent ranking changes not reflected yet              |
| 7    | refresh_content | Stale page with opportunity for refresh                  | Search demand may have shifted                        |
| 8    | refresh_content | Very high visibility (25715 impressions) but low CTR     | Position may recently improve without clicks updating |
| 9    | refresh_content | Old content and low CTR                                  | Data quality issue in click tracking                  |
| 10   | refresh_content | Stale and visible content                                | Content refresh already scheduled                     |


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Pick 1

content_074ba6ead17b

Only 533 impressions despite a score of 5.
The page may not generate enough traffic to justify a refresh.

Weak Pick 2:
content_bdbec75c1148

CTR is low, but impressions are only 1316.
The opportunity may be smaller than the score suggests.

Weak Pick 3:
content_e3ff1b093148

Moderate visibility but no evidence that refreshing content would improve performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.